In [1]:
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

/home/4284/.conda/envs/rse/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = "/lustre/darse/users/4284/rse_resourses/1400"
BATCH_SIZE = 16
LATENT_DIM = 64
EPOCHS = 1000
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class TiffDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(
            glob.glob(os.path.join(folder, "*.tif")) +
            glob.glob(os.path.join(folder, "*.tiff"))
        )
        self.transform_dino = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),  # DINO expects 3 channels
            transforms.ToTensor(),
        ])
        self.transform_target = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("L")


        #getting the original width and height
        w, h = img.size

        #to ensure the image is actually taller than 62 pixels before cropping
        if h > 62:
            # Crop takes (left, top, right, bottom)
            # We keep from 0 to w (width) and 0 to (h - 62) (height without the panel)
            img = img.crop((0, 0, w, h - 62))
        # ----------------------

        x_dino = self.transform_dino(img)     # [3,224,224]
        x_target = self.transform_target(img) # [1,64,64]
        return x_dino, x_target

In [4]:

dataset = TiffDataset(DATA_DIR)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

/home/4284/.conda/envs/rse/lib/python3.11/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:

#loading DINOv2 encoder

dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dino.eval().to(DEVICE)

for p in dino.parameters():
    p.requires_grad = False

# figure out DINO embedding dim
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
    dummy_feat = dino(dummy)
    DINO_DIM = dummy_feat.shape[-1]

print("DINO embedding dim:", DINO_DIM)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /tmp/torch/hub/main.zip
/tmp/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/tmp/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/tmp/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /tmp/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|████████████████████████████████████████████████████████████████████████████████████████████████| 84.2M/84.2M [00:00<00:00, 115MB/s]


DINO embedding dim: 384


In [6]:
# VAE on top of DINO features
class DinoVAE(nn.Module):
    def __init__(self, dino_dim=DINO_DIM, latent_dim=LATENT_DIM):
        super().__init__()

        # VAE encoder from DINO features -> latent params
        self.fc_mu = nn.Sequential(
            nn.Linear(dino_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.fc_logvar = nn.Sequential(
            nn.Linear(dino_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

        # latent -> image decoder
        self.decoder_input = nn.Linear(latent_dim, 256 * 8 * 8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    # 64x64
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def encode(self, feat):
        mu = self.fc_mu(feat)
        logvar = self.fc_logvar(feat)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(-1, 256, 8, 8)
        return self.decoder(x)

    def forward(self, feat):
        mu, logvar = self.encode(feat)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

model = DinoVAE().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [7]:
def vae_loss(recon, target, mu, logvar, beta=1e-3):
    #recon_loss = F.mse_loss(recon, target, reduction="mean")
    recon_loss = F.l1_loss(recon, target, reduction="mean") #for enhancing the edge detection
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100
)

lr_history = []
loss_history = []

In [ ]:
# Training
beta_max = 1e-3
warmup_epochs = 20

loss_history = []
recon_history = []
kl_history = []
beta_history = []
lr_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    beta = min(beta_max, (epoch + 1) / warmup_epochs * beta_max)

    for x_dino, x_target in loader:
        x_dino = x_dino.to(DEVICE)
        x_target = x_target.to(DEVICE)

        with torch.no_grad():
            feat = dino(x_dino)

        recon, mu, logvar = model(feat)

        loss, recon_loss, kl = vae_loss(
            recon, x_target, mu, logvar, beta=beta
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl.item()

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    lr_history.append(current_lr)

    avg_loss = total_loss / len(loader)
    avg_recon = total_recon / len(loader)
    avg_kl = total_kl / len(loader)

    loss_history.append(avg_loss)

    loss_history.append(avg_loss)
    recon_history.append(avg_recon)
    kl_history.append(avg_kl)
    beta_history.append(beta)
    lr_history.append(current_lr)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss {avg_loss:.4f} | "
        f"Recon {avg_recon:.4f} | "
        f"KL {avg_kl:.4f} | "
        f"Beta {beta:.6f} | "
        f"LR {current_lr:.6f}"
    )


Epoch 001 | Loss 0.0544 | Recon 0.0544 | KL 1.5637 | Beta 0.000050 | LR 0.001000
Epoch 002 | Loss 0.0472 | Recon 0.0471 | KL 1.2405 | Beta 0.000100 | LR 0.000999
Epoch 003 | Loss 0.0458 | Recon 0.0456 | KL 1.0094 | Beta 0.000150 | LR 0.000998
Epoch 004 | Loss 0.0453 | Recon 0.0451 | KL 0.7486 | Beta 0.000200 | LR 0.000996
Epoch 005 | Loss 0.0450 | Recon 0.0449 | KL 0.6070 | Beta 0.000250 | LR 0.000994
Epoch 006 | Loss 0.0450 | Recon 0.0448 | KL 0.4767 | Beta 0.000300 | LR 0.000991
Epoch 007 | Loss 0.0449 | Recon 0.0448 | KL 0.4005 | Beta 0.000350 | LR 0.000988
Epoch 008 | Loss 0.0448 | Recon 0.0447 | KL 0.3684 | Beta 0.000400 | LR 0.000984
Epoch 009 | Loss 0.0447 | Recon 0.0446 | KL 0.3285 | Beta 0.000450 | LR 0.000980
Epoch 010 | Loss 0.0448 | Recon 0.0446 | KL 0.3190 | Beta 0.000500 | LR 0.000976
Epoch 011 | Loss 0.0446 | Recon 0.0444 | KL 0.3022 | Beta 0.000550 | LR 0.000970
Epoch 012 | Loss 0.0446 | Recon 0.0444 | KL 0.2842 | Beta 0.000600 | LR 0.000965
Epoch 013 | Loss 0.0445 | Re

In [ ]:
# to show reconstructions
model.eval()
x_dino, x_target = next(iter(loader))
x_dino = x_dino.to(DEVICE)
x_target = x_target.to(DEVICE)

with torch.no_grad():
    feat = dino(x_dino)
    recon, _, _ = model(feat)

x_target = x_target.cpu()
recon = recon.cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(6):
    axes[0, i].imshow(x_target[i, 0], cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title("Real")

    axes[1, i].imshow(recon[i, 0], cmap="gray")
    axes[1, i].axis("off")
    axes[1, i].set_title("Recon")
plt.tight_layout()
plt.show()

# =========================================================
# Generate NEW images
# =========================================================
with torch.no_grad():
    z = torch.randn(12, LATENT_DIM).to(DEVICE)
    samples = model.decode(z).cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(12):
    axes[i // 6, i % 6].imshow(samples[i, 0], cmap="gray")
    axes[i // 6, i % 6].axis("off")
plt.tight_layout()
plt.show()